# SHAP

Runs the **SHAP package** (`shap.TreeExplainer`) across all datasets and task types
in the `woodelfhd_depth_sweep_experiment`.  
This notebook is **independent** and can run in parallel with the other method notebooks.

### What this notebook does
1. Mounts Google Drive (results are saved there after each mission)
2. Clones `treebranchmarks` repo
3. Installs all dependencies
4. Runs `woodelfhd_depth_sweep_experiment --method shap`
5. Writes partial results to Drive as `shap.json`

### Datasets (all download automatically)
| Dataset | Source |
|---------|--------|
| Fraud Detection | Google Drive parquet (~200 MB) |
| HIGGS | Google Drive parquet |
| KDD Cup (Intrusion Detection) | Google Drive parquet |
| California Housing | sklearn builtin |

### SHAPApproach behaviour in this experiment
- `background_shap_interactions`: not supported by the shap library — recorded as `not_supported`.
- `background_shap`: uses `bg_shap_limit=10` background rows (configured in the experiment);
  times are extrapolated when the full background set is larger.
- `woodelf_explainer` is still needed (it is a dependency of `treebranchmarks`).

In [ ]:
# ── Step 1: Mount Google Drive ──────────────────────────────────────────────
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [ ]:
# ── Step 2: Configure paths ──────────────────────────────────────────────────
# Must match the DRIVE_FOLDER used in notebooks 01, 02, and 04.
import pathlib

DRIVE_FOLDER = pathlib.Path('/content/drive/MyDrive/ShapResearch/HighDepth/treebranchmark_experiments/results_jsons')
DRIVE_FOLDER.mkdir(parents=True, exist_ok=True)

DRIVE_RESULT_PATH = DRIVE_FOLDER / 'shap.json'
print(f'Results will be saved to: {DRIVE_RESULT_PATH}')

Results will be saved to: /content/drive/MyDrive/ShapResearch/HighDepth/treebranchmark_experiments/results_jsons/shap.json


In [ ]:
# ── Step 3: Clone repository ───────────────────────────────────────────────
TREEBRANCHMARKS_URL = 'https://github.com/ron-wettenstein/TreeBranchMarks.git'  # e.g. https://github.com/ron-wettenstein/WoodelfExperiments.git

!git clone {TREEBRANCHMARKS_URL} /content/treebranchmarks

Cloning into '/content/treebranchmarks'...
remote: Enumerating objects: 363, done.
remote: Counting objects: 100% (363/363), done.
remote: Compressing objects: 100% (216/216), done.
remote: Total 363 (delta 230), reused 267 (delta 137), pack-reused 0 (from 0)
Receiving objects: 100% (363/363), 220.06 KiB | 16.93 MiB/s, done.
Resolving deltas: 100% (230/230), done.


In [ ]:
# ── Step 4: Install packages ─────────────────────────────────────────────────
# woodelf_explainer must be installed before treebranchmarks (it is listed
# as a dependency in treebranchmarks/pyproject.toml).

!pip install woodelf_explainer

!pip install -q -e /content/treebranchmarks

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 41.2/41.2 kB 3.3 MB/s eta 0:00:00
  Installing build dependencies ... done
  Checking if build backend supports build_editable ... done
  Getting requirements to build editable ... done
  Preparing editable metadata (pyproject.toml) ... done
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.8/1.8 MB 61.4 MB/s eta 0:00:00
  Building editable for treebranchmarks (pyproject.toml) ... done


In [ ]:
# ── Step 5: Restore method cache from a previous interrupted run ─────────────
import shutil, pathlib

cache_dir = pathlib.Path('/content/treebranchmarks/cache/method_results/woodelfhd_depth_sweep_experiment')
cache_dir.mkdir(parents=True, exist_ok=True)
local_cache_file = cache_dir / 'shap.json'

if DRIVE_RESULT_PATH.exists() and not local_cache_file.exists():
    shutil.copy(DRIVE_RESULT_PATH, local_cache_file)
    print(f'Restored method cache ({DRIVE_RESULT_PATH.stat().st_size // 1024} KB)')
else:
    print('No method cache to restore — starting fresh.')

Restored method cache (57 KB)


In [ ]:
# ── Step 6: Run the experiment (SHAP only) ───────────────────────────────────
# --method shap : only the SHAPApproach is timed
# The method name 'shap' matches SHAP.name in builtin.py.

%cd /content/treebranchmarks

!python -u -m benchmarks.woodelfhd_depth_sweep_experiment \
    --method shap \
    --result_location "{DRIVE_RESULT_PATH}"

/content/treebranchmarks

Mission: fraud_detection PD SHAP sweep_D
  dataset   : fraud_detection
  D_values  : [6, 9, 12, 15, 18, 21]
  n=118108  m=0
  tasks     : ['Path-Dependent SHAP']
[dataset:fraud_detection] Cache miss — downloading and preprocessing.
Downloading...
From: https://drive.google.com/uc?id=1A1Qdtron9XtZ6h85uNdaFCprUkH5KA5P
To: /content/treebranchmarks/cache/datasets/fraud_detection/raw/data.parquet
100% 69.6M/69.6M [00:00<00:00, 134MB/s]
[dataset:fraud_detection] Cached 590540 rows × 397 features.

  > D=6  n=118108  m=0
[model:lightgbm] Training.
[model:lightgbm] Trained in 15.43s — T=100, D=6, L=32.0, F=397
  [approach:SHAP] CACHED=14.012s

  > D=9  n=118108  m=0
[model:lightgbm] Training.
[model:lightgbm] Trained in 18.03s — T=100, D=9, L=82.8, F=397
  [approach:SHAP] CACHED=66.733s

  > D=12  n=118108  m=0
[model:lightgbm] Training.
[model:lightgbm] Trained in 21.68s — T=100, D=12, L=154.0, F=397
  [approach:SHAP] CACHED=186.604s

  > D=15  n=118108  m=0
[model:l

In [ ]:
# ── Step 7: Verify output ────────────────────────────────────────────────────
import json

with open(DRIVE_RESULT_PATH) as f:
    cache = json.load(f)

print(f'Entries in method cache: {len(cache)}')
if cache:
    sample = next(iter(cache.values()))
    print(f'Sample entry: {sample["_label"]}  →  {sample["running_time"]:.3f}s')
print(f'\nFile saved to: {DRIVE_RESULT_PATH}')

Entries in method cache: 104
Sample entry: Path-Dependent SHAP n=118108 m=0 D=6 T=100  →  14.012s

File saved to: /content/drive/MyDrive/ShapResearch/HighDepth/treebranchmark_experiments/results_jsons/shap.json


# Old Runs

In [ ]:
# ── Step 6: Run the experiment (SHAP only) ───────────────────────────────────
# --method shap : only the SHAPApproach is timed
# The method name 'shap' matches SHAP.name in builtin.py.

%cd /content/treebranchmarks

!python -u -m benchmarks.woodelfhd_depth_sweep_experiment \
    --method shap \
    --result_location "{DRIVE_RESULT_PATH}"

/content/treebranchmarks

Mission: fraud_detection PD SHAP sweep_D
  dataset   : fraud_detection
  D_values  : [6, 9, 12, 15, 18, 21]
  n=118108  m=0
  tasks     : ['Path-Dependent SHAP']
[dataset:fraud_detection] Cache miss — downloading and preprocessing.
Downloading...
From: https://drive.google.com/uc?id=1A1Qdtron9XtZ6h85uNdaFCprUkH5KA5P
To: /content/treebranchmarks/cache/datasets/fraud_detection/raw/data.parquet
100% 69.6M/69.6M [00:00<00:00, 218MB/s]
[dataset:fraud_detection] Cached 590540 rows × 397 features.

  > D=6  n=118108  m=0
[model:lightgbm] Training.
[model:lightgbm] Trained in 11.49s — T=100, D=6, L=32.0, F=397
  [approach:SHAP] CACHED=14.012s

  > D=9  n=118108  m=0
[model:lightgbm] Training.
[model:lightgbm] Trained in 13.91s — T=100, D=9, L=82.8, F=397
  [approach:SHAP] CACHED=66.733s

  > D=12  n=118108  m=0
[model:lightgbm] Training.
[model:lightgbm] Trained in 17.19s — T=100, D=12, L=154.0, F=397
  [approach:SHAP] CACHED=186.604s

  > D=15  n=118108  m=0
[model:l

In [ ]:
# ── Step 6: Run the experiment (SHAP only) ───────────────────────────────────
# --method shap : only the SHAPApproach is timed
# The method name 'shap' matches SHAP.name in builtin.py.

%cd /content/treebranchmarks

!python -u -m benchmarks.woodelfhd_depth_sweep_experiment \
    --method shap \
    --result_location "{DRIVE_RESULT_PATH}"

/content/treebranchmarks

Mission: fraud_detection PD SHAP sweep_D
  dataset   : fraud_detection
  D_values  : [6, 9, 12, 15, 18, 21]
  n=118108  m=0
  tasks     : ['Path-Dependent SHAP']
[dataset:fraud_detection] Cache miss — downloading and preprocessing.
Downloading...
From: https://drive.google.com/uc?id=1A1Qdtron9XtZ6h85uNdaFCprUkH5KA5P
To: /content/treebranchmarks/cache/datasets/fraud_detection/raw/data.parquet
100% 69.6M/69.6M [00:00<00:00, 214MB/s]
[dataset:fraud_detection] Cached 590540 rows × 397 features.

  > D=6  n=118108  m=0
[model:lightgbm] Training.
[model:lightgbm] Trained in 10.91s — T=100, D=6, L=32.0, F=397
  [approach:SHAP] CACHED=14.012s

  > D=9  n=118108  m=0
[model:lightgbm] Training.
[model:lightgbm] Trained in 13.15s — T=100, D=9, L=82.8, F=397
  [approach:SHAP] CACHED=66.733s

  > D=12  n=118108  m=0
[model:lightgbm] Training.
[model:lightgbm] Trained in 15.64s — T=100, D=12, L=154.0, F=397
  [approach:SHAP] CACHED=186.604s

  > D=15  n=118108  m=0
[model:l